In [3]:
from google.colab import files
uploaded = files.upload()


Saving Sample - Superstore.csv to Sample - Superstore.csv


In [4]:
import pandas as pd

df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
print("Shape (rows, columns):", df.shape)
print()
print("Column names:")
print(df.columns.tolist())
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

Shape (rows, columns): (9994, 21)

Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Missing values per column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Duplicate rows: 0


In [6]:
dupes_meaningful = df.duplicated(subset=["Order ID", "Product ID"]).sum()
print("Duplicate Order ID + Product ID combos:", dupes_meaningful)

Duplicate Order ID + Product ID combos: 8


In [7]:
before = len(df)
df = df.drop_duplicates(subset=["Order ID", "Product ID"], keep="first")
after = len(df)

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Duplicates removed: {before - after}")

Rows before: 9994
Rows after: 9986
Duplicates removed: 8


In [8]:
print("Missing values per column:")
print(df.isna().sum())

Missing values per column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [9]:
invalid_sales = (df["Sales"] <= 0).sum()
invalid_qty = (df["Quantity"] <= 0).sum()

print("Rows with invalid Sales (<=0):", invalid_sales)
print("Rows with invalid Quantity (<=0):", invalid_qty)

Rows with invalid Sales (<=0): 0
Rows with invalid Quantity (<=0): 0


In [10]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")

print(df["Order Date"].dtype)
print("Earliest date:", df["Order Date"].min())
print("Latest date:", df["Order Date"].max())

datetime64[ns]
Earliest date: 2014-01-03 00:00:00
Latest date: 2017-12-30 00:00:00


In [11]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)

df["Margin"] = df["Profit"] / df["Sales"]

df[["Order Date", "Year", "Month", "YearMonth", "Sales", "Profit", "Margin"]].head()

,Order Date,Year,Month,YearMonth,Sales,Profit,Margin
0,2016-11-08,2016,11,2016-11,261.9600,41.9136,0.1600
1,2016-11-08,2016,11,2016-11,731.9400,219.5820,0.3000
2,2016-06-12,2016,6,2016-06,14.6200,6.8714,0.4700
3,2015-10-11,2015,10,2015-10,957.5775,-383.0310,-0.4000
4,2015-10-11,2015,10,2015-10,22.3680,2.5164,0.1125


In [12]:
by_category = df.groupby("Category").agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_margin=("Margin", "mean"),
    margin_std=("Margin", "std"),
    n_orders=("Order ID", "count")
).reset_index()

by_category

,Category,total_sales,total_profit,avg_margin,margin_std,n_orders
0,Furniture,741432.0433,18380.2814,0.038704,0.344644,2119
1,Office Supplies,718317.7920,122247.4038,0.137843,0.548070,6022
2,Technology,835759.7370,145386.1344,0.156131,0.230684,1845


In [13]:
by_region = df.groupby("Region").agg(
    total_sales=("Sales", "sum"),
    avg_margin=("Margin", "mean"),
    n_orders=("Order ID", "count")
).reset_index()

by_region

,Region,total_sales,avg_margin,n_orders
0,Central,501239.8908,-0.104073,2323
1,East,677906.3680,0.167151,2845
2,South,391007.8250,0.163059,1616
3,West,725355.4885,0.219512,3202


In [14]:
by_subcat = df.groupby("Sub-Category").agg(
    total_sales=("Sales", "sum"),
    avg_margin=("Margin", "mean"),
    margin_std=("Margin", "std"),
    n_orders=("Order ID", "count")
).reset_index()

by_subcat.sort_values("n_orders")

,Sub-Category,total_sales,avg_margin,margin_std,n_orders
6,Copiers,149528.0300,0.317194,0.121400,68
11,Machines,189238.6310,-0.072026,0.557960,115
15,Supplies,46673.5380,0.112039,0.180024,190
8,Fasteners,3024.2800,0.299171,0.191825,217
4,Bookcases,114879.9963,-0.126640,0.469470,228
7,Envelopes,16476.4020,0.423140,0.065225,254
16,Tables,206965.5320,-0.147727,0.276850,319
10,Labels,12486.3120,0.429663,0.064447,364
1,Appliances,107532.1610,-0.156869,0.982089,466
5,Chairs,328167.7310,0.044040,0.154011,616


In [15]:
df.to_csv("clean_superstore.csv", index=False)
by_category.to_csv("category_kpis.csv", index=False)
by_region.to_csv("region_kpis.csv", index=False)
by_subcat.to_csv("subcategory_kpis.csv", index=False)

from google.colab import files
files.download("clean_superstore.csv")
files.download("category_kpis.csv")
files.download("region_kpis.csv")
files.download("subcategory_kpis.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.2 MB/s eta 0:00:00


In [18]:
from groq import Groq

client = Groq(api_key="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")

In [19]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working."}
    ]
)

print(response.choices[0].message.content)

Hello. I'm working and ready to assist you with any questions or tasks you may have. How can I help you today?


In [20]:
# Reload your KPI data (in case the notebook reset)
import pandas as pd
by_category = pd.read_csv("category_kpis.csv")

# Turn the KPI table into plain text the AI can read
kpi_text = by_category.to_string(index=False)
print(kpi_text)

       Category  total_sales  total_profit  avg_margin  margin_std  n_orders
      Furniture  741432.0433    18380.2814    0.038704    0.344644      2119
Office Supplies  718317.7920   122247.4038    0.137843    0.548070      6022
     Technology  835759.7370   145386.1344    0.156131    0.230684      1845


In [21]:
prompt = f"""
You are a business analyst reviewing sales data. Here is a summary of sales
performance by product category:

{kpi_text}

Based on this data, write 2-3 short, confident business insights or
predictions about these categories. Write as if you are certain about your
conclusions.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

ai_insight = response.choices[0].message.content
print(ai_insight)

Based on the sales data, it's clear that Office Supplies is our most profitable category, with a total profit of $122,247.40 and an average margin of 13.78%. This is significantly higher than the other two categories, and I predict that continued investment in this area will yield substantial returns. In fact, I'm confident that Office Supplies will continue to drive growth and account for an increasingly large share of our overall profit.

In contrast, Furniture is a lower-margin category, with an average margin of just 3.87%. While it still generates significant sales volume, the lower profitability means that we should focus on optimizing our operations and supply chain to maximize efficiency and minimize costs in this area. I'm certain that by streamlining our Furniture business, we can improve its overall contribution to our bottom line.

Finally, Technology shows promise as a high-margin category, with an average margin of 15.63%. However, its relatively low sales volume and stan

In [22]:
# Reload the full KPI tables we need
by_category = pd.read_csv("category_kpis.csv")
by_subcat = pd.read_csv("subcategory_kpis.csv")

print(by_category)

          Category  total_sales  total_profit  avg_margin  margin_std  \
0        Furniture  741432.0433    18380.2814    0.038704    0.344644   
1  Office Supplies  718317.7920   122247.4038    0.137843    0.548070   
2       Technology  835759.7370   145386.1344    0.156131    0.230684   

   n_orders  
0      2119  
1      6022  
2      1845  


In [24]:
def check_volatility_confidence(category_name, by_category_df):
    """
    Checks how volatile a category's margin is, and returns a confidence
    label + plain-English reason.
    """
    row = by_category_df[by_category_df["Category"] == category_name].iloc[0]
    std = row["margin_std"]

    # Thresholds based on what we actually see in this dataset
    # (Technology ~0.23 is our most stable, Office Supplies ~0.55 is our least)
    if std < 0.28:
        confidence = "High"
        reason = f"Margin volatility is low (std={std:.3f}), meaning this category's profitability is relatively stable and predictable."
    elif std < 0.45:
        confidence = "Medium"
        reason = f"Margin volatility is moderate (std={std:.3f}), meaning some caution is warranted before acting on forecasts."
    else:
        confidence = "Low"
        reason = f"Margin volatility is high (std={std:.3f}), meaning this category's profit swings significantly — any confident claim about it should be treated with skepticism."

    return confidence, reason

# Test it on all 3 categories
for cat in by_category["Category"]:
    conf, reason = check_volatility_confidence(cat, by_category)
    print(f"{cat}: {conf} confidence")
    print(f"   Reason: {reason}\n")

Furniture: Medium confidence
   Reason: Margin volatility is moderate (std=0.345), meaning some caution is warranted before acting on forecasts.

Office Supplies: Low confidence
   Reason: Margin volatility is high (std=0.548), meaning this category's profit swings significantly — any confident claim about it should be treated with skepticism.

Technology: High confidence
   Reason: Margin volatility is low (std=0.231), meaning this category's profitability is relatively stable and predictable.



In [25]:
def check_sample_size_confidence(name, df, name_col="Category", min_orders=500):
    """
    Checks if there's enough data behind a claim to trust it.
    """
    row = df[df[name_col] == name].iloc[0]
    n = row["n_orders"]

    if n >= min_orders:
        confidence = "High"
        reason = f"Based on a large sample ({n} orders), giving reasonable statistical reliability."
    elif n >= 150:
        confidence = "Medium"
        reason = f"Based on a moderate sample ({n} orders) — reasonably reliable, but not as robust as a larger sample."
    else:
        confidence = "Low"
        reason = f"Based on a small sample ({n} orders) — conclusions here are less statistically reliable and could shift with more data."

    return confidence, reason

# Test on sub-categories (more variation in sample size here)
for subcat in by_subcat["Sub-Category"]:
    conf, reason = check_sample_size_confidence(subcat, by_subcat, name_col="Sub-Category")
    print(f"{subcat}: {conf} confidence — {reason}")

Accessories: High confidence — Based on a large sample (773 orders), giving reasonable statistical reliability.
Appliances: Medium confidence — Based on a moderate sample (466 orders) — reasonably reliable, but not as robust as a larger sample.
Art: High confidence — Based on a large sample (796 orders), giving reasonable statistical reliability.
Binders: High confidence — Based on a large sample (1522 orders), giving reasonable statistical reliability.
Bookcases: Medium confidence — Based on a moderate sample (228 orders) — reasonably reliable, but not as robust as a larger sample.
Chairs: High confidence — Based on a large sample (616 orders), giving reasonable statistical reliability.
Copiers: Low confidence — Based on a small sample (68 orders) — conclusions here are less statistically reliable and could shift with more data.
Envelopes: Medium confidence — Based on a moderate sample (254 orders) — reasonably reliable, but not as robust as a larger sample.
Fasteners: Medium confiden

In [26]:
def trust_gap_score(name, kpi_df, name_col="Category", min_orders=500):
    """
    Combines volatility + sample size into one overall confidence score.
    Returns: overall confidence label, list of reasons, and a warning if
    the AI is likely to sound more confident than the data supports.
    """
    row = kpi_df[kpi_df[name_col] == name].iloc[0]
    std = row["margin_std"]
    n = row["n_orders"]

    reasons = []
    score = 0  # higher score = more trustworthy

    # Volatility signal
    if std < 0.28:
        score += 2
        reasons.append(f"Low margin volatility (std={std:.3f}) — stable, predictable profit.")
    elif std < 0.45:
        score += 1
        reasons.append(f"Moderate margin volatility (std={std:.3f}) — some fluctuation.")
    else:
        reasons.append(f"High margin volatility (std={std:.3f}) — profit swings significantly.")

    # Sample size signal
    if n >= min_orders:
        score += 2
        reasons.append(f"Large sample ({n} orders) — statistically reliable.")
    elif n >= 150:
        score += 1
        reasons.append(f"Moderate sample ({n} orders) — reasonably reliable.")
    else:
        reasons.append(f"Small sample ({n} orders) — less statistically reliable.")

    # Final verdict
    if score >= 3:
        overall = "High"
    elif score >= 1:
        overall = "Medium"
    else:
        overall = "Low"

    return overall, reasons

# Test on all sub-categories, sorted worst to best
results = []
for subcat in by_subcat["Sub-Category"]:
    overall, reasons = trust_gap_score(subcat, by_subcat, name_col="Sub-Category")
    results.append({"Sub-Category": subcat, "Overall_Confidence": overall, "Reasons": " | ".join(reasons)})

results_df = pd.DataFrame(results)
results_df

,Sub-Category,Overall_Confidence,Reasons
0,Accessories,High,"Low margin volatility (std=0.161) — stable, pr..."
1,Appliances,Medium,High margin volatility (std=0.982) — profit sw...
2,Art,High,"Low margin volatility (std=0.107) — stable, pr..."
3,Binders,Medium,High margin volatility (std=0.773) — profit sw...
4,Bookcases,Medium,High margin volatility (std=0.469) — profit sw...
5,Chairs,High,"Low margin volatility (std=0.154) — stable, pr..."
6,Copiers,Medium,"Low margin volatility (std=0.121) — stable, pr..."
7,Envelopes,High,"Low margin volatility (std=0.065) — stable, pr..."
8,Fasteners,High,"Low margin volatility (std=0.192) — stable, pr..."
9,Furnishings,High,Moderate margin volatility (std=0.377) — some ...


In [27]:
category_results = []
for cat in by_category["Category"]:
    overall, reasons = trust_gap_score(cat, by_category, name_col="Category")
    category_results.append({"Category": cat, "Trust_Gap_Verdict": overall, "Reasons": " | ".join(reasons)})

category_trust_df = pd.DataFrame(category_results)
category_trust_df

,Category,Trust_Gap_Verdict,Reasons
0,Furniture,High,Moderate margin volatility (std=0.345) — some ...
1,Office Supplies,Medium,High margin volatility (std=0.548) — profit sw...
2,Technology,High,"Low margin volatility (std=0.231) — stable, pr..."


In [28]:
comparison = pd.DataFrame({
    "Category": ["Furniture", "Office Supplies", "Technology"],
    "AI_Claim_Summary": [
        "Lower-margin category; needs cost optimization to improve profitability.",
        "AI stated: 'confident' this category will drive continued growth.",
        "AI stated: needs 'careful management' due to volatility concerns."
    ],
    "Trust_Gap_Verdict": category_trust_df["Trust_Gap_Verdict"].values,
    "Actual_Data_Reason": category_trust_df["Reasons"].values
})

comparison

,Category,AI_Claim_Summary,Trust_Gap_Verdict,Actual_Data_Reason
0,Furniture,Lower-margin category; needs cost optimization...,High,Moderate margin volatility (std=0.345) — some ...
1,Office Supplies,AI stated: 'confident' this category will driv...,Medium,High margin volatility (std=0.548) — profit sw...
2,Technology,AI stated: needs 'careful management' due to v...,High,"Low margin volatility (std=0.231) — stable, pr..."


In [29]:
comparison.to_csv("trust_gap_comparison.csv", index=False)
category_trust_df.to_csv("category_trust_scores.csv", index=False)
results_df.to_csv("subcategory_trust_scores.csv", index=False)

from google.colab import files
files.download("trust_gap_comparison.csv")
files.download("category_trust_scores.csv")
files.download("subcategory_trust_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>